# Experiment 1 Colab Runner

This notebook is built for interactive phase-by-phase work in both local Jupyter/VS Code and Google Colab.

Why this notebook is structured this way:
- one startup block selects the environment and prepares paths
- one cell loads or updates the repo when needed
- one cell defines the experiment config directly in Colab
- one cell previews synthetic training targets before training
- one cell runs a single phase
- one cell inspects saved metrics/examples for that phase
- one optional cell runs the full experiment loop

This makes it easier to debug one phase at a time instead of running the whole pipeline blindly.

In [ ]:
import sys
from datetime import datetime

print('Notebook stdout sanity check')
print('UTC time:', datetime.utcnow().isoformat())
print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])
sys.stdout.flush()

## Runner setup

Run the next cells in order.

If the sanity cell above prints nothing, the issue is your notebook kernel or VS Code/Colab runtime, not the training code.

Use `RUN_ENV = 'local'` for VS Code/Jupyter on your computer and `RUN_ENV = 'colab'` for Google Colab.

In [ ]:
# Set RUN_ENV to 'local' on your computer or 'colab' in Google Colab.
# Leave AUTO_RUN_ENV=True if you want automatic detection from the notebook runtime.
AUTO_RUN_ENV = True
RUN_ENV = 'colab'

GIT_REPO_URL = 'https://github.com/ffbskt/AgentAI.git'
GIT_BRANCH = 'rlvr-simplify-colab'
LOCAL_REPO_DIR = r'D:/Codex projects/Transformer_Graph3/arithmetic-transformer'
COLAB_PROJECT_ROOT = '/content/drive/MyDrive/AgentAI'
RUN_NAME = 'colab_run'

try:
    import google.colab  # type: ignore
    detected_env = 'colab'
except ImportError:
    detected_env = 'local'

if AUTO_RUN_ENV:
    RUN_ENV = detected_env

assert RUN_ENV in {'local', 'colab'}, f'Unsupported RUN_ENV: {RUN_ENV}'
print('Detected env:', detected_env)
print('Selected env:', RUN_ENV)
print('Requested git branch:', GIT_BRANCH)

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
else:
    print('Local mode selected, skipping Google Drive mount')

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def ensure_repo_ready(run_env: str) -> tuple[Path, Path, Path]:
    if run_env == 'colab':
        project_root = Path(COLAB_PROJECT_ROOT)
        repo_root = project_root
        git_dir = repo_root / '.git'

        if git_dir.exists():
            print(f"'{repo_root}' is already a git repository. Fetching branch '{GIT_BRANCH}'...")
            subprocess.run(['git', '-C', str(repo_root), 'fetch', 'origin', GIT_BRANCH], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'checkout', GIT_BRANCH], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'reset', '--hard', f'origin/{GIT_BRANCH}'], check=True)
        elif repo_root.exists():
            print(f"'{repo_root}' exists but is not a git repository. Removing and re-cloning...")
            shutil.rmtree(repo_root)
            project_root.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, str(repo_root)], check=True)
        else:
            print(f"Cloning branch '{GIT_BRANCH}' from '{GIT_REPO_URL}' into '{repo_root}'...")
            project_root.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, str(repo_root)], check=True)

        current_branch = subprocess.check_output(['git', '-C', str(repo_root), 'branch', '--show-current'], text=True).strip()
        current_commit = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
        print('Checked out branch:', current_branch)
        print('Checked out commit:', current_commit)
        assert current_branch == GIT_BRANCH, f'Expected branch {GIT_BRANCH}, got {current_branch}'

        repo_dir = repo_root / 'arithmetic-transformer'
    else:
        repo_dir = Path(LOCAL_REPO_DIR)
        if not repo_dir.exists():
            raise FileNotFoundError(f'Local repo dir does not exist: {repo_dir}')
        print(f'Using local repo without clone: {repo_dir}')

    experiment_dir = repo_dir / 'experiment_1'
    log_root = experiment_dir / 'logs' / RUN_NAME
    log_root.mkdir(parents=True, exist_ok=True)
    return repo_dir, experiment_dir, log_root

REPO_DIR, EXPERIMENT_DIR, LOG_ROOT = ensure_repo_ready(RUN_ENV)

print('Repo dir:', REPO_DIR)
print('Experiment dir:', EXPERIMENT_DIR)
print('Log root:', LOG_ROOT)
assert REPO_DIR.exists(), f'Repo dir does not exist: {REPO_DIR}'
assert EXPERIMENT_DIR.exists(), f'Experiment dir does not exist: {EXPERIMENT_DIR}'

In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
requirements_file = REPO_DIR / 'requirements-colab.txt'
assert requirements_file.exists(), f'Missing requirements file: {requirements_file}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)], check=True)
print('Requirements installed for', RUN_ENV)

## Editable experiment config

Why keep config in a notebook cell:
- fast changes in Colab without editing repo files
- easy to try a different phase set or training budget
- the Python runner accepts this config as inline JSON

In [ ]:
import json
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PHASE_OUTPUT_ROOT = str(LOG_ROOT)

EXPERIMENT_CONFIG = {
    'model': {
        'kind': 'transformer-sine',
        'hidden_size': 64,
        'ffw_size': 128,
        'num_layers': 2,
        'num_heads': 4,
        'dropout': 0.0,
        'lr': 0.001
    },
    'train_defaults': {
        'epochs': 1,
        'batch_size': 64,
        'train_samples': 256,
        'val_samples': 128,
        'test_samples': 128,
        'max_new_tokens': 48,
        'rl_steps': 4,
        'rl_batch_size': 16,
        'num_generations': 4,
        'temperature': 1.0,
        'kl_coef': 0.02,
        'best_of_n_steps': 2,
        'seed': 11
    },
    'phases': [
        {'name': 'step1_1d', 'description': '1-digit warmup', 'fmt': 'A', 'shape': '1d+1d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step2_2d', 'description': '2-digit reasoning', 'fmt': 'C', 'shape': '2d+2d', 'carry_mode': 'simple', 'rl_enabled': True},
        {'name': 'step3_3d', 'description': '3-digit reasoning', 'fmt': 'C', 'shape': '3d+3d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step4_4d', 'description': '4-digit reasoning', 'fmt': 'C', 'shape': '4d+4d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step5_5d', 'description': '5-digit reasoning', 'fmt': 'C', 'shape': '5d+5d', 'carry_mode': 'heavy', 'rl_enabled': True}
    ]
}

CONFIG_JSON = json.dumps(EXPERIMENT_CONFIG)
print('Selected device:', DEVICE)
print('Phase output root:', PHASE_OUTPUT_ROOT)
print('Configured phases:', [phase['name'] for phase in EXPERIMENT_CONFIG['phases']])
assert len(EXPERIMENT_CONFIG['phases']) > 0, 'EXPERIMENT_CONFIG must contain at least one phase'

## Import the phase-oriented runner functions

Why these functions exist:
- `load_experiment_config`: build usable phase objects from notebook config
- `preview_phase_samples`: inspect expected symbolic targets before training
- `run_one_phase`: train and evaluate one phase only
- `run_experiment`: loop over all phases when you want the full pipeline

In [ ]:
import importlib
import os
import sys
from pathlib import Path

os.chdir(EXPERIMENT_DIR)
print('Working directory:', Path.cwd())

# Remove stale notebook imports so Colab/VS Code does not keep an older module object.
sys.modules.pop('train_experiment_1', None)
train_experiment_1 = importlib.import_module('train_experiment_1')
train_experiment_1 = importlib.reload(train_experiment_1)

print('Imported module file:', train_experiment_1.__file__)
available_symbols = [
    name for name in ['load_experiment_config', 'preview_phase_samples', 'run_one_phase', 'run_experiment']
    if hasattr(train_experiment_1, name)
]
print('Available runner symbols:', available_symbols)
assert len(available_symbols) == 4, 'Notebook loaded an unexpected train_experiment_1 version'

load_experiment_config = train_experiment_1.load_experiment_config
preview_phase_samples = train_experiment_1.preview_phase_samples
run_one_phase = train_experiment_1.run_one_phase
run_experiment = train_experiment_1.run_experiment

raw_config, phases = load_experiment_config(config_json=CONFIG_JSON)
print('Loaded phases:', [phase.name for phase in phases])
assert len(phases) > 0, 'No phases were loaded from the experiment config'

## Preview one phase before training

This is useful because RLVR depends on deterministic target style.
If the generated symbolic targets already look wrong here, training will also be wrong.

In [ ]:
PHASE_INDEX = 0  # Change this to inspect a different phase.
assert 0 <= PHASE_INDEX < len(phases), f'PHASE_INDEX {PHASE_INDEX} out of range for {len(phases)} phases'
preview = preview_phase_samples(phases[PHASE_INDEX], sample_count=5)
print('Preview sample count:', len(preview))
assert len(preview) > 0, 'Preview returned no samples'
for item in preview:
    print(item)
    print('-' * 80)


## Run one phase only

This cell is the most important one for debugging.
Use it when you want to train exactly one phase, save its outputs, and inspect them before continuing.

In [ ]:
import torch

PHASE_INDEX = 0  # Change this to run another phase.
assert 0 <= PHASE_INDEX < len(phases), f'PHASE_INDEX {PHASE_INDEX} out of range for {len(phases)} phases'
phase = phases[PHASE_INDEX]
phase_output_dir = Path(PHASE_OUTPUT_ROOT) / phase.name
print('Running phase:', phase.name)
print('Device:', DEVICE)
print('Output dir:', phase_output_dir)

model = None
model, summary = run_one_phase(
    phase=phase,
    model=model,
    device=torch.device(DEVICE),
    output_root=Path(PHASE_OUTPUT_ROOT),
    sample_limit=32,
)
assert (phase_output_dir / 'summary.json').exists(), 'summary.json was not created'
summary

## Inspect saved phase artifacts

Keep training and inspection separate.
This helps answer questions like:
- did SFT improve anything?
- did RLVR help or hurt?
- what examples is the model producing?

In [ ]:
import json

phase_dir = Path(PHASE_OUTPUT_ROOT) / phases[PHASE_INDEX].name
print('Phase dir:', phase_dir)
assert phase_dir.exists(), f'Phase output dir does not exist: {phase_dir}'
print('\nSummary:')
print(json.loads((phase_dir / 'summary.json').read_text(encoding='utf-8')))

print('\nPost-SFT examples:')
print(json.loads((phase_dir / 'post_sft_examples.json').read_text(encoding='utf-8'))[:2])

print('\nPost-RLVR examples:')
print(json.loads((phase_dir / 'post_rlvr_examples.json').read_text(encoding='utf-8'))[:2])

## Run the full experiment loop

Use this only after single-phase checks look sane.
It runs all configured phases in order and writes a top-level summary.

In [ ]:
summaries = run_experiment(
    config_json=CONFIG_JSON,
    device=DEVICE,
    output_dir=PHASE_OUTPUT_ROOT,
)
assert len(summaries) > 0, 'Full experiment produced no phase summaries'
summaries

In [ ]:
import subprocess

print('Artifacts in', PHASE_OUTPUT_ROOT)
for path in sorted(Path(PHASE_OUTPUT_ROOT).iterdir()):
    print(path)

summary_csv = Path(PHASE_OUTPUT_ROOT) / 'summary.csv'
if summary_csv.exists():
    print('\nsummary.csv contents:')
    print(summary_csv.read_text(encoding='utf-8'))
else:
    print('\nsummary.csv does not exist yet. Run the full experiment cell first if you need it.')